In [19]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone

In [ ]:
def preprocess_joplin_2011(filepath):
    """
    Reads the Joplin 2011 dataset.
    Columns: num, text, timestamp, label
    Timestamp is in day/month/year format (24/04/2011).
    label is numeric but uncertain. We'll assume 1=on-topic, 0=off-topic.
    We'll create an artificial 'tweet_id' since it's not provided.
    """
    # Reading CSV
    joplin_df = pd.read_csv(filepath, sep=",", names=["num","text","timestamp","label"], header=0)
    
    # Creating a tweet_id from 'num' / running index
    joplin_df["tweet_id"] = "joplin_" + joplin_df["num"].astype(str)

    # Parsing the timestamp as dd/mm/yyyy
    # Some might be just day/month/year, so handling that carefully.
    joplin_df["timestamp"] = pd.to_datetime(joplin_df["timestamp"], format="%d/%m/%Y", errors="coerce")
    
    # Renaming columns to unify with other sets
    joplin_df.rename(columns={
        "text": "text",
        "label": "label"
    }, inplace=True)
    
    # Converting label to int (if needed, and if it's numeric string)
    # If 'label' is already numeric, just ensuring it's int
    joplin_df["label"] = joplin_df["label"].astype(int, errors="ignore")
    
    # Creating a 'user_handle' column / a placeholder
    joplin_df["user_handle"] = np.nan  # Not provided in Joplin data

    # Dropping columns we don't need
    # 'num' is no longer needed, we used it for tweet_id
    joplin_df.drop(columns=["num"], inplace=True)

    # 8. Reordering columns:
    final_cols = ["tweet_id", "text", "timestamp", "label", "user_handle"]
    joplin_df = joplin_df[final_cols]

    return joplin_df


In [21]:
TW_EPOCH = 1288834974657  # 2010‑11‑04 01:42:54.657 UTC

def id_to_datetime(snowflake: int) -> pd.Timestamp:
    """
    Deterministically recover the tweet‑creation time from a 64‑bit
    Twitter snowflake ID. Returns a tz‑naïve pandas.Timestamp.
    """
    # stripping lower 22 bits
    millis = (snowflake >> 22) + TW_EPOCH        
    return (pd.to_datetime(millis, unit="ms", utc=True)
              .tz_localize(None))
    # dropping tz for consistency

In [22]:
def preprocess_oklahoma_2013(filepath):
    """
    Reads the Oklahoma 2013 dataset (Excel or CSV).
    Columns: 'tweet id', 'tweet', 'label'
    No timestamp. We'll set timestamp = NaN
    'label' is 'on-topic' or 'off-topic'
    We'll unify to numeric: on-topic=1, off-topic=0.
    """
    oklahoma_df = pd.read_csv(filepath)
    
    oklahoma_df.columns = oklahoma_df.columns.str.strip()
    oklahoma_df.rename(columns=str.strip, inplace=True)
    # 1. Renaming columns to unify
    oklahoma_df.rename(columns={
        "tweet id": "tweet_id",
        "tweet": "text",
        "label": "label"
    }, inplace=True)
    
    # 3. Clean & cast tweet_id  →  int64
    oklahoma_df["tweet_id"] = (
        oklahoma_df["tweet_id"]
        .astype(str)
        .str.strip("'\"")        # remove single‑/double‑quotes
        .astype("int64", errors="raise")
    )

    # 4. Recover timestamp from snowflake
    oklahoma_df["timestamp"] = oklahoma_df["tweet_id"].apply(id_to_datetime)
    
    # Converting label = "on-topic" => 1, "off-topic" => 0
    def label_to_num(x):
        if isinstance(x, str):
            return 1 if x.lower() == "on-topic" else 0
        return 0
    oklahoma_df["label"] = oklahoma_df["label"].apply(label_to_num)
    
    # Creating 'user_handle' = np.nan since not provided
    oklahoma_df["user_handle"] = np.nan

    # Reordering columns
    final_cols = ["tweet_id", "text", "timestamp", "label", "user_handle"]
    oklahoma_df = oklahoma_df[final_cols]
    
    return oklahoma_df

In [23]:
def preprocess_2022_scrape(filepath):
    """
    Reads the 2022 dataset that includes columns like:
    Name, Handle, Timestamp, Verified, Content, Comments, Retweets, Likes...
    We'll unify to 'tweet_id', 'text', 'timestamp', 'label', 'user_handle'.
    There's no 'label' here, so we might set label = 1 or keep it empty.
    """
    # Loading CSV
   
    df_2022 = pd.read_csv(filepath)
    
    # 2. Renaming columns
    # - "Content" -> "text"
    # - "Tweet ID" -> "tweet_id"
    # - "Timestamp" is already a recognized name
    df_2022.rename(columns={
        "Content": "text",
        "Tweet ID": "tweet_id",
        "Timestamp": "timestamp",
        "Handle": "user_handle"
    }, inplace=True)

    # 3. Parsing 'timestamp' (ISO 8601) => Python datetime
    df_2022["timestamp"] = pd.to_datetime(df_2022["timestamp"], errors="coerce")
    
    # 4. We don't have a known label column => set it to np.nan or "1" if we want them all 'on-topic'
    # For demonstration, let's set them to label=1 as "general tornado" tweets
    df_2022["label"] = 1 
    
    # Keeping only the relevant columns we want in the final merge
    keep_cols = ["tweet_id", "text", "timestamp", "label", "user_handle"]
    # If some columns don't exist, filtering them carefully
    existing_cols = [c for c in keep_cols if c in df_2022.columns]
    df_2022_final = df_2022[existing_cols].copy()
    
    # 6. Converting tweet_id to string in case it is numeric
    df_2022_final["tweet_id"] = df_2022_final["tweet_id"].astype(str)
    
    return df_2022_final

In [25]:
def preprocess_kentucky_2021(filepath):
    """
    Kentucky tornado‑related tweets (Dec‑2021) scraped with columns:
    Name, Handle, Timestamp, Verified, Content, … , Tweet ID
    We keep the same unified columns as the other sets.
    """
    ky_df = pd.read_csv(filepath)

    ky_df.rename(columns={
        "Content"   : "text",
        "Tweet ID"  : "tweet_id",
        "Timestamp" : "timestamp",
        "Handle"    : "user_handle"}, inplace=True)

    ky_df["timestamp"] = pd.to_datetime(ky_df["timestamp"], errors="coerce")
    ky_df["label"]     = 1                       # mark as on‑topic
    ky_df["tweet_id"]  = ky_df["tweet_id"].astype(str)

    return ky_df[["tweet_id","text","timestamp","label","user_handle"]]

In [26]:
def merge_all_datasets(joplin_path, ok_path, data2022_path, ky2021_path):
    """
    Returning a consolidated DataFrame with columns:
    tweet_id • text • timestamp • label • user_handle
    """
    dfs = [
        preprocess_joplin_2011(joplin_path),
        preprocess_oklahoma_2013(ok_path),
        preprocess_2022_scrape(data2022_path),
        preprocess_kentucky_2021(ky2021_path)
    ]

    merged = pd.concat(dfs, ignore_index=True)
    merged.drop(columns=["user_handle"], inplace=True, errors="ignore")
    
    merged.drop_duplicates(subset=["tweet_id","text"], keep="first", inplace=True)
    merged["timestamp"] = pd.to_datetime(merged["timestamp"], utc=True, errors="coerce")
    merged["timestamp"] = merged["timestamp"].dt.tz_localize(None)   # strip tz → all tz‑naive
    merged.sort_values(by="timestamp", inplace=True, na_position="last")

    return merged.reset_index(drop=True)

In [30]:
# DRIVER
if __name__ == "__main__":
    joplin_fp   = "/Users/muhammadfaizanraza/Desktop/PSGVSpring2/NLP_Project/Joplin_2011_tweets.csv"
    ok_fp       = "/Users/muhammadfaizanraza/Desktop/PSGVSpring2/NLP_Project/2013_Oklahoma_Tornado-ontopic_offtopic.csv"
    scrape22_fp = "/Users/muhammadfaizanraza/Desktop/PSGVSpring2/NLP_Project/Tornado-X 1.csv"
    ky2021_fp   = "/Users/muhammadfaizanraza/Desktop/PSGVSpring2/NLP_Project/Kentucky_Tornado_2021_tweets.csv"   # <‑‑ new path

    final_df = merge_all_datasets(joplin_fp, ok_fp, scrape22_fp, ky2021_fp)

    final_df = final_df.dropna()
    final_df.columns
    print(final_df.head(25))
    print("TOTAL TWEETS AFTER MERGE:", len(final_df))

    final_df.to_csv("/Users/muhammadfaizanraza/Desktop/PSGVSpring2/NLP_Project/merged_tornado_tweets.csv", index=False)

       tweet_id                                               text  timestamp  \
0    joplin_998  @morganjpalmer and I just ran for our lives. I... 2011-04-20   
1   joplin_1044  Would you sit in your car and make a video of ... 2011-04-20   
3   joplin_1046  My car got destroyed in the tornado but nothin... 2011-04-20   
4   joplin_1050  Tornado Takes Burger, Fries & Drink RIGHT Out ... 2011-04-20   
5   joplin_1078  if theres a tornado why the fuck would you get... 2011-04-20   
6   joplin_1088  Just got in my car and realized my windows wer... 2011-04-20   
7   joplin_1089  Tornado Takes Burger, Fries & Drink RIGHT Out ... 2011-04-20   
8   joplin_1117  Welcome to tornado season everyone:  Did you k... 2011-04-20   
9   joplin_1118  Can you guess what I hate today? Late nights, ... 2011-04-20   
10  joplin_1119  Why a Car Does Not Give Good Protection in a T... 2011-04-20   
11  joplin_1120  Gonna send color books and crayons and some ca... 2011-04-20   
12  joplin_1121  Lol I asked